# Testing with pytest

Writing tests is how you prove your code works — and keep it working as it changes.

**Install:** `pip install pytest pytest-cov pytest-asyncio`

**Run:** `pytest -v` or `pytest --cov=src --cov-report=term-missing`

**In this notebook:**
- Your first test
- Assertions and `pytest.approx`
- `pytest.raises` — testing exceptions
- Fixtures
- `@pytest.mark.parametrize`
- `monkeypatch`
- Mocking with `unittest.mock`
- Coverage

## 1. Your First Test

pytest discovers tests automatically in files named `test_*.py` or `*_test.py`. Test functions must start with `test_`.

In [ ]:
# Code under test
def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError('Cannot divide by zero')
    return a / b

# Tests — normally in test_calculator.py
# Shown here for demonstration; run with pytest, not directly
def test_add_positive():
    assert add(2, 3) == 5

def test_add_negative():
    assert add(-1, -1) == -2

# Quick sanity check (not how pytest works in practice)
test_add_positive()
test_add_negative()
print('Tests passed!')

## 2. Assertions and pytest.approx

pytest rewrites `assert` statements to show detailed failure messages. Never use `==` for floats — use `pytest.approx` instead.

In [ ]:
import pytest

# Float comparison — why == fails
print(0.1 + 0.2)          # 0.30000000000000004
print(0.1 + 0.2 == 0.3)   # False!

# pytest.approx uses a relative tolerance of 1e-6 by default
print(0.1 + 0.2 == pytest.approx(0.3))        # True
print(0.1 + 0.2 == pytest.approx(0.3, rel=1e-3))   # True with 0.1% tolerance

# Works with lists and dicts too
result = [0.1 + 0.2, 0.3 + 0.4]
print(result == pytest.approx([0.3, 0.7]))   # True

## 3. pytest.raises — Testing Exceptions

Use `pytest.raises` as a context manager to verify that the correct exception is raised.

In [ ]:
import pytest

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError('Cannot divide by zero')
    return a / b

# Basic — just check the exception type
with pytest.raises(ZeroDivisionError):
    divide(10, 0)
print('ZeroDivisionError raised correctly')

# With match — check the message (regex)
with pytest.raises(ZeroDivisionError, match='Cannot divide'):
    divide(10, 0)
print('Message matched correctly')

# Access the exception object
with pytest.raises(ZeroDivisionError) as exc_info:
    divide(10, 0)
print(f'Exception: {exc_info.value}')
print(f'Type: {exc_info.type}')

## 4. Fixtures

Fixtures provide reusable test setup (and teardown). They are injected by name as function parameters.

```python
@pytest.fixture
def my_fixture():
    # setup
    value = SomeClass()
    yield value   # test runs here
    # teardown — runs after test, even on failure
    value.cleanup()
```

In [ ]:
import pytest

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner    = owner
        self._balance = balance

    def deposit(self, amount):
        if amount <= 0: raise ValueError('Must be positive')
        self._balance += amount

    def balance(self):
        return self._balance

# --- In a test file ---
@pytest.fixture
def account():
    return BankAccount('Alice', 1000)

@pytest.fixture
def empty_account():
    return BankAccount('Bob', 0)

def test_deposit(account):          # receives the fixture result
    account.deposit(500)
    assert account.balance() == 1500

def test_deposit_invalid(account):
    with pytest.raises(ValueError):
        account.deposit(-100)

# Run manually for notebook demo
acc = BankAccount('Alice', 1000)
acc.deposit(500)
print(f'Balance: {acc.balance()}')   # 1500

## 5. @pytest.mark.parametrize

Run the same test logic with multiple input sets — avoids copy-pasting test bodies.

In [ ]:
import pytest

def fizzbuzz(n):
    if n % 15 == 0: return 'FizzBuzz'
    if n % 3 == 0:  return 'Fizz'
    if n % 5 == 0:  return 'Buzz'
    return n

# In a test file, this generates 8 individual test cases:
# test_fizzbuzz[1-1], test_fizzbuzz[3-Fizz], etc.
@pytest.mark.parametrize('n, expected', [
    (1,  1),
    (3,  'Fizz'),
    (5,  'Buzz'),
    (9,  'Fizz'),
    (10, 'Buzz'),
    (15, 'FizzBuzz'),
    (30, 'FizzBuzz'),
    (7,  7),
])
def test_fizzbuzz(n, expected):
    assert fizzbuzz(n) == expected

# Verify our function manually
cases = [(3,'Fizz'),(5,'Buzz'),(15,'FizzBuzz'),(7,7)]
for n, exp in cases:
    result = fizzbuzz(n)
    status = '✓' if result == exp else '✗'
    print(f'{status} fizzbuzz({n}) = {result!r}')

## 6. Monkeypatching

`monkeypatch` (a built-in fixture) temporarily replaces attributes, environment variables, or functions during a test — restores the original state after the test.

In [ ]:
import os

def get_greeting():
    name = os.environ.get('USER_NAME', 'stranger')
    return f'Hello, {name}!'

# In a test file:
# def test_custom_name(monkeypatch):
#     monkeypatch.setenv('USER_NAME', 'Alice')
#     assert get_greeting() == 'Hello, Alice!'
#
# def test_default_name(monkeypatch):
#     monkeypatch.delenv('USER_NAME', raising=False)
#     assert get_greeting() == 'Hello, stranger!'

# Quick demo without pytest:
os.environ['USER_NAME'] = 'Alice'
print(get_greeting())   # Hello, Alice!
del os.environ['USER_NAME']
print(get_greeting())   # Hello, stranger!

## 7. Mocking with unittest.mock

`Mock` objects replace real dependencies (HTTP clients, SMTP servers, databases) in tests. They record all calls made to them and can return preset values.

In [ ]:
from unittest.mock import Mock, patch

class EmailService:
    def __init__(self, smtp):
        self._smtp = smtp

    def send_welcome(self, email, name):
        return self._smtp.send(
            to=email, subject='Welcome!', body=f'Hi {name}!'
        )

# Create a mock SMTP client — no real server needed
mock_smtp = Mock()
mock_smtp.send.return_value = {'status': 'sent'}

service = EmailService(mock_smtp)
result = service.send_welcome('alice@test.com', 'Alice')

print(f'Result: {result}')   # {'status': 'sent'}

# Verify the mock was called correctly
mock_smtp.send.assert_called_once_with(
    to='alice@test.com', subject='Welcome!', body='Hi Alice!'
)
print(f'Call count: {mock_smtp.send.call_count}')   # 1
print('All assertions passed!')

In [ ]:
# @patch replaces an object for the duration of a test
from unittest.mock import patch

def fetch_user(user_id):
    import requests
    response = requests.get(f'https://api.example.com/users/{user_id}')
    return response.json()

# In a test file:
# @patch('requests.get')
# def test_fetch_user(mock_get):
#     mock_get.return_value.json.return_value = {'id': 1, 'name': 'Alice'}
#     result = fetch_user(1)
#     assert result == {'id': 1, 'name': 'Alice'}
#     mock_get.assert_called_once_with('https://api.example.com/users/1')

print('Mock patterns demonstrated above')

## 8. Coverage

```bash
pip install pytest-cov
pytest --cov=src --cov-report=term-missing
```

Output shows which lines were NOT executed during tests:

```
Name          Stmts   Miss  Cover   Missing
-------------------------------------------
src/bank.py      45      3    93%   72-74
```

**Aim for high coverage on business logic.** 100% coverage doesn't mean bug-free — it means every line was executed, not every edge case was tested.

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | assert, fixtures, pytest.raises |
| [02-medium.py](exercises/02-medium.py) | Medium | parametrize, tmp_path, module scope |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | Mock, monkeypatch, AsyncMock |

Solutions: [solutions/](solutions/)